# Week 08 — Multi-Agent Systems with LangGraph
_Nawaloka Hospital AI — Declarative Graph-Based Orchestration_

---

This notebook covers the **Week 08 upgrade**: replacing the imperative Python orchestrator with a **LangGraph StateGraph** — the production-grade foundation for multi-agent AI systems.

## Why LangGraph?

| Problem with Week 07 (imperative loop) | LangGraph solution |
|---|---|
| State passed as function arguments — implicit, fragile | Typed `AgentState` — explicit, validated |
| Single synthesiser adapts via prompt | Separate persona nodes — true specialization |
| `if/elif` routing in `_dispatch()` | Compiled conditional edges |
| No visualization | `draw_mermaid()` out of the box |
| Hard to add nodes — must refactor `chat()` | `add_node()` + `add_edge()` |
| No parallelism | Fan-out edges (future-ready) |

## What You Will Learn

| Section | Topic |
|---|---|
| **1** | Build & visualize the LangGraph StateGraph |
| **2** | `AgentState` — the shared conveyor belt |
| **3** | Supervisor-Worker pattern explained |
| **4** | Direct Agent — greetings & web search |
| **5** | Clinical Agent — RAG + structured memory injection |
| **6** | Admin Agent — CRM operations |
| **7** | Structured memory: `semantic_facts` as `list[dict]` |
| **8** | Full multi-turn conversation (all routes) |
| **9** | State inspection — what each node saw |
| **10** | LangFuse: per-node traces |

## Graph Topology

```
[START]
   │
   ▼
[recall_node]         ← ST turns + LT facts → state.memory_context + state.semantic_facts
   │
   ▼
[supervisor_node]     ← QueryRouter.route() → state.route_decision
   │
   ▼  (conditional edge: supervisor_routing)
   ├── 'admin'    → [admin_agent_node]    (CRM: bookings, patient lookup)
   ├── 'clinical' → [clinical_agent_node] (RAG: hospital KB, medical facts)
   └── 'direct'   → [direct_agent_node]  (greetings, web search, concierge)
                          │
                          ▼
                  [save_memory_node]      ← ST write + conditional LT distillation
                          │
                        [END]
```

## Prerequisites
- Complete Notebook 01 (or `make seed-crm-xl` + `make ingest-qdrant`)
- `.env` configured with all API keys

---
## Section 0 — Setup

In [ ]:
import sys, os, re
sys.path.insert(0, "../src")

from dotenv import load_dotenv
load_dotenv()

from infrastructure.log import setup_logging
from loguru import logger
setup_logging("INFO", for_notebook=True)

import pandas as pd
from sqlalchemy.orm import sessionmaker
from infrastructure.db import get_sql_engine
from infrastructure.db.crm_models import Patient, Booking, Doctor, Location, Specialty

logger.success("Environment ready")

---
## Section 1 — Build & Visualize the Graph

One of the biggest advantages of LangGraph is that the agent's logic **is a graph** — and we can visualise it instantly without reading any code.

In [ ]:
from agents.orchestrator import build_agent
from langchain_core.messages import HumanMessage

agent = build_agent(enable_crm=True, enable_rag=True, enable_web=True)

logger.success("LangGraph agent built")
logger.info(f"  CRM tool        : {'✓' if agent.crm_tool else '✗'}")
logger.info(f"  RAG tool        : {'✓' if agent.rag_tool else '✗'}")
logger.info(f"  Web search tool : {'✓' if agent.web_tool else '✗'}")

In [ ]:
# ── Visualize the StateGraph ─────────────────────────────────────
# draw_mermaid_png() renders via the Mermaid.ink API and returns PNG bytes.
# We wrap it in IPython.display.Image so Jupyter shows it inline.

from IPython.display import Image, display

graph_png = agent.graph.get_graph(xray=True).draw_mermaid_png()
display(Image(graph_png))

# Also print the raw Mermaid text (useful for docs / slides)
print("\nRaw Mermaid (paste into https://mermaid.live):")
print(agent.graph.get_graph(xray=True).draw_mermaid())

In [ ]:
# ── Inspect registered nodes and edges programmatically ─────────
# Each edge is a NamedTuple: Edge(source, target, data, conditional)
g = agent.graph.get_graph()

print("Nodes:")
for node_id in g.nodes:
    print(f"  • {node_id}")

print("\nEdges:")
for edge in g.edges:
    kind = " [conditional]" if edge.conditional else ""
    label = f" ({edge.data})" if edge.data else ""
    print(f"  {edge.source:30s} → {edge.target}{label}{kind}")

---
## Section 2 — AgentState: The Shared Conveyor Belt

In Week 07, state was implicit — passed as function arguments and return values. In Week 08, state is an **explicit TypedDict** that every node reads from and writes to:

```python
class AgentState(TypedDict):
    messages:       Annotated[list[AnyMessage], add_messages]  # LangChain message accumulator
    user_id:        str
    session_id:     str
    memory_context: Optional[str]         # ST turns as formatted text
    semantic_facts: Optional[list[dict]]  # ← KEY UPGRADE: LT facts as structured dicts
    route_decision: Optional[dict]        # {route, action, params, reasoning, confidence}
    tool_output:    Optional[str]
    final_answer:   Optional[str]
    should_distill: Optional[bool]
```

### The Critical Upgrade: `semantic_facts`

| Week 07 | Week 08 |
|---|---|
| `memory_context: str` — all memory collapsed to one string | `semantic_facts: list[dict]` — structured, each sub-agent decides what to use |
| Clinical and admin agents get same blob | Clinical agent injects ALL medical facts; admin agent filters for CRM-relevant ones |
| No selective memory per persona | Each node independently composes its own context window |

### How State Flows

```
initial state = {user_id, session_id, messages: [HumanMessage]}
     │
     ▼ recall_node
{..., memory_context: "=== RECENT ...", semantic_facts: [{text, tags, score}, ...]}
     │
     ▼ supervisor_node
{..., route_decision: {route: "clinical", action: None, params: {}, confidence: 0.95}}
     │
     ▼ clinical_agent_node
{..., tool_output: "Evidence from KB...", final_answer: "Based on Nawaloka's protocols..."}
     │
     ▼ save_memory_node  →  [END]
```

In [ ]:
from agents.state import AgentState
import inspect

print("AgentState fields:")
print("=" * 72)
hints = AgentState.__annotations__
for field, annotation in hints.items():
    print(f"  {field:20s} : {annotation}")

---
## Section 3 — The Supervisor-Worker Pattern

Week 07 had **one general synthesiser** that adapted its tone via prompt. Week 08 introduces a true **Supervisor-Worker** architecture:

```
               [supervisor_node]
               QueryRouter.route()
               route_decision → {route, confidence, action, params}
                     │
         ┌───────────┼───────────┐
         ▼           ▼           ▼
   [admin_agent]  [clinical_agent]  [direct_agent]
```

### Sub-Agent Personas and Guardrails

| Node | Persona | Tools | Guardrails |
|---|---|---|---|
| `admin_agent_node` | CRM Coordinator — professional, scheduling-focused | `CRMTool` | Refuses medical questions. Only CRM actions. |
| `clinical_agent_node` | Clinical Knowledge — evidence-based, patient-history aware | `RAGTool` | No diagnosis. Cites KB sources. Injects ALL LT medical facts. |
| `direct_agent_node` | Concierge — welcoming, helpful | `WebSearchTool` (fallback) | Greetings + general queries. Falls back to Tavily for live info. |

The supervisor (`supervisor_routing` edge function) maps:
- `route == "crm"` → `"admin"`
- `route == "rag"` → `"clinical"`
- `route in {"direct", "web_search"}` → `"direct"`

---
## Section 4 — Demo: Direct Agent (Graph Routing in Action)

In Week 07, a greeting hit an `if/elif` chain inside `_dispatch()`. Now, watch how **LangGraph's conditional edges** handle the same message:

1. `recall_node` → loads memory context from ST + LT stores  
2. `supervisor_node` → LLM classifies intent → returns `route_decision = "direct"`  
3. **Conditional edge fires** → routes to `direct_agent_node`  
4. `save_memory_node` → persists the turn  

The Direct Agent handles:
- Greetings and identity messages
- General conversational queries
- Web search fallback (when the question needs live data via Tavily)


In [ ]:
def extract_phone(text: str) -> str:
    match = re.search(r"\+?[\d][\d\s\-\.\(\)]{7,18}[\d]", text)
    if not match:
        raise ValueError("No phone number found.")
    raw = re.sub(r"\D", "", match.group())
    if raw.startswith("0") and len(raw) == 10:
        raw = "94" + raw[1:]
    elif len(raw) == 9 and not raw.startswith("94"):
        raw = "94" + raw
    return raw


def show(resp, label=""):
    """Print an AgentResponse with clear section headers."""
    print("\n" + "\u2550" * 72)
    if label:
        print(f"  {label}")
    print("\u2550" * 72)
    route_str = resp.route.upper()
    if resp.action:
        route_str += f" / {resp.action}"
    # Show all routes if multi-route
    if hasattr(resp, 'routes') and len(resp.routes) > 1:
        route_str += f"  (multi-route: {', '.join(r.upper() for r in resp.routes)})"
    print(f"  Route      : {route_str}")
    print(f"  Latency    : {resp.latency_ms}ms")
    if hasattr(resp, 'routes') and len(resp.routes) > 1:
        print(f"  Fan-out    : {len(resp.routes)} agents ran in parallel")
    if resp.memory_context and resp.memory_context.strip():
        lines = resp.memory_context.strip().split("\n")
        print(f"  Memory ctx : {len(lines)} lines")
        for ln in lines[:6]:
            print(f"    {ln}")
    if resp.tool_output and resp.tool_output.strip():
        print(f"  Tool output:")
        for ln in resp.tool_output.strip().split("\n")[:5]:
            print(f"    {ln}")
    print(f"\n  Answer:\n    {resp.answer}")
    print("\u2550" * 72)


logger.success("Helpers ready")

In [ ]:
# The greeting message carries the user's identity (phone number)
greeting = "Hi! I'm Anushka. My mobile is +94 78 10 30 736."
USER_ID    = extract_phone(greeting)
SESSION_ID = "nb02-demo"

logger.info(f"USER_ID = {USER_ID}")

# ── Turn 1: Direct Agent ────────────────────────────────────────
resp = agent.chat(user_message=greeting, user_id=USER_ID, session_id=SESSION_ID)
show(resp, "DIRECT AGENT — greeting")

# Verify the supervisor's route decision
print(f"\nRoute decision: {resp.route}")
print("The supervisor classified this as 'direct' — no tool needed.")
print("LangGraph edge: supervisor_node → direct_agent_node → save_memory_node")

In [ ]:
# ── Web search fallback: Direct Agent calls Tavily ──────────────
resp_web = agent.chat(
    user_message="What are the current visiting hours at Nawaloka Hospital?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show(resp_web, "DIRECT AGENT — web_search route (Tavily)")

print("\nNote: route='web_search' → still handled by direct_agent_node")
print("The Direct Agent persona decides when to use Tavily vs. answer directly.")

---
## Section 5 — Demo: Clinical Agent (Selective State Injection)

This is the **key architectural upgrade** from Week 07. In the imperative version, memory was collapsed into one flat string and the single synthesiser got everything.

In LangGraph, `AgentState` carries `semantic_facts` as a **structured `list[dict]`**. The Clinical Agent node selectively injects only medically-relevant facts (tagged `allergy`, `medication`, `condition`) into its system prompt — the Admin Agent never sees these.

**Watch the data flow:**
1. `recall_node` → populates `state["semantic_facts"]` with structured dicts  
2. `supervisor_node` → classifies as `rag` → conditional edge fires to `clinical_agent_node`  
3. `clinical_agent_node` → filters facts by medical tags, calls RAG tool, synthesises answer  
4. `save_memory_node` → triggers distillation, extracts new LT facts  


In [ ]:
# First, give the agent some medical context to remember
resp_mem = agent.chat(
    user_message="I'm allergic to penicillin — please always remember this. I also take atenolol 50mg daily.",
    user_id=USER_ID, session_id=SESSION_ID,
)
show(resp_mem, "DIRECT AGENT — memory trigger ('always remember')")
print("\nDistiller now extracts 'penicillin allergy' and 'atenolol 50mg' as LT facts.")

In [ ]:
# Clinical question → routes to clinical_agent_node
resp_rag = agent.chat(
    user_message="What cardiac diagnostic procedures and services does Nawaloka Hospital offer?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show(resp_rag, "CLINICAL AGENT — RAG + patient-specific memory")

print("\nObserve:")
print("  • Route = rag → supervisor routes to clinical_agent_node")
print("  • Clinical Agent retrieved 'penicillin allergy' from semantic_facts")
print("  • KB search via Qdrant provided the cardiology department details")
print("  • Answer combines KB evidence with patient-specific memory")

In [ ]:
# Same semantic intent → CAG cache HIT (instant answer)
resp_cached = agent.chat(
    user_message="Tell me about the cardiology department services.",
    user_id=USER_ID, session_id=SESSION_ID,
)
show(resp_cached, "CLINICAL AGENT — RAG (CAG semantic cache HIT)")

stats = agent.rag_tool.cache_stats()
print(f"\nCAG Cache stats:")
print(f"  Entries    : {stats['total_cached']}")
print(f"  Threshold  : {stats['similarity_threshold']}")
print(f"  Backend    : {stats['backend']}")
print(f"  Collection : {stats['collection']}")
print("'cardiology department services' semantically matched cached query → instant response")

---
## Section 6 — Demo: Admin Agent (CRM Dispatch via Graph Edges)

In Week 07, CRM dispatch was a method call inside `_dispatch()`. Now it's a **separate graph node** with its own persona and guardrails.

The Admin Agent:
- Receives `route_decision` with the extracted `action` and `params` from the supervisor
- Dispatches to `CRMTool` for patient/doctor/booking operations
- Has strict guardrails: **refuses medical questions** (redirects to Clinical Agent)
- Speaks in a professional, scheduling-coordinator tone

| CRM Action | What it does |
|---|---|
| `lookup_patient` | Search by phone / name / patient_id |
| `search_doctors` | Filter by specialty / location / name |
| `create_booking` | Create appointment (with conflict checking) |
| `cancel_booking` | Cancel by booking_id |
| `reschedule_booking` | Reschedule with conflict checking |

**The graph advantage:** Adding a new CRM action = updating the tool, not refactoring the orchestrator.


In [ ]:
# Patient lookup → routes to admin_agent_node
resp_crm = agent.chat(
    user_message="Can you look up my appointments? My phone is +94 78 10 30 736.",
    user_id=USER_ID, session_id=SESSION_ID,
)
show(resp_crm, "ADMIN AGENT — CRM lookup_patient")

print("\nLangGraph edge: supervisor_node → admin_agent_node → save_memory_node")

In [ ]:
# Doctor search → routes to admin_agent_node, action=search_doctors
resp_docs = agent.chat(
    user_message="Which cardiologists are available at Nawaloka Hospital?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show(resp_docs, "ADMIN AGENT — CRM search_doctors (specialty=cardiology)")

# Verify against actual DB
session = sessionmaker(bind=get_sql_engine())()
try:
    cardiologists = (
        session.query(Doctor, Specialty)
        .join(Specialty, Doctor.specialty_id == Specialty.specialty_id)
        .filter(Specialty.name.ilike("%cardiol%"), Doctor.active == 1)
        .limit(5).all()
    )
    if cardiologists:
        df = pd.DataFrame([{
            "Doctor": f"Dr. {d.full_name}", "Specialty": s.name, "Phone": d.phone or "—"
        } for d, s in cardiologists])
        print("\nDB verification (Supabase → doctors table):")
        display(df)
finally:
    session.close()

---
## Section 7 — Structured Memory: `semantic_facts` as `list[dict]`

This is the most important architectural difference from Week 07.

### Week 07 (collapsed string):
```python
# Memory collapsed to one flat string BEFORE any agent sees it
memory_context = recaller.format_context(st_turns, lt_facts)  # → str
# The single synthesiser gets everything — no selectivity
```

### Week 08 (structured list):
```python
# recall_node sets BOTH fields on AgentState
state['memory_context'] = recaller.format_context(st_turns, lt_facts)  # str (display)
state['semantic_facts'] = [fact.__dict__ for fact in lt_facts]          # list[dict]

# clinical_agent_node selectively injects medical facts:
medical_facts = [
    f for f in state['semantic_facts']
    if any(tag in f.get('tags', []) for tag in ['allergy', 'medication', 'condition'])
]
# admin_agent_node filters for scheduling-relevant facts
```

In [ ]:
# Directly query the long-term store to see what structured facts exist
from infrastructure.llm import get_default_embeddings
from memory import LongTermMemoryStore

embedder = get_default_embeddings()
lt_store = LongTermMemoryStore(embedder)

# Recall facts relevant to clinical context
clinical_facts = lt_store.query(USER_ID, "medications allergies health conditions", k=5, threshold=0.2)

print(f"LT facts for user {USER_ID} (structured list[dict] in AgentState):\n")
print(f"{'─' * 72}")
for i, f in enumerate(clinical_facts, 1):
    print(f"Fact {i}:")
    print(f"  text  : {f.text}")
    print(f"  tags  : {f.tags}")
    print(f"  score : {f.score:.3f}")
    print()

print("In AgentState, these travel as:")
structured = [{"text": f.text, "tags": f.tags, "score": f.score} for f in clinical_facts]
import json
print(json.dumps(structured[:2], indent=2))
print(f"\n→ Clinical Agent injects all allergy/medication facts into its system prompt")
print(f"→ Admin Agent ignores medical tags, looks only for appointment-relevant facts")

---
## Section 8 — Full Multi-Turn Conversation (All Sub-Agents)

A complete 7-turn conversation that exercises **all three sub-agents** in sequence. This is where the graph architecture shines — watch:

1. **Memory accumulates** across turns (context grows with each iteration)  
2. **Supervisor routes correctly** to different persona nodes each turn  
3. **Clinical Agent** uses patient-specific facts from earlier turns (selective injection)  
4. **Admin Agent** stays in CRM-coordinator mode (persona guardrails)  
5. **Each turn traverses the same graph** — but takes different conditional edges  


In [ ]:
FULL_SESSION = "nb02-full-demo"
first = "Hello! I'm Anushka and I'd like some help today. My mobile number is +94 78 10 30 736."
FULL_USER = extract_phone(first)

turns = [
    (first,                                                                    "direct"),
    ("I take atenolol 50mg daily for blood pressure — please remember this.",  "direct"),
    ("I'm allergic to penicillin. Very important — always remember!",           "direct"),
    ("Which cardiologists are available at Nawaloka Hospital?",                 "crm"),
    ("What is the cardiac rehabilitation protocol at the hospital?",            "rag"),
    ("What do you remember about my medications and allergies?",                "direct"),
    ("What are the current visiting hours at Nawaloka?",                        "web_search"),
]

print("Full Multi-Turn Conversation — All Sub-Agents")
print("=" * 72)

for i, (msg, expected_route) in enumerate(turns, 1):
    resp = agent.chat(user_message=msg, user_id=FULL_USER, session_id=FULL_SESSION)
    
    ctx_lines = len(resp.memory_context.strip().split("\n")) if resp.memory_context.strip() else 0
    match = "✓" if resp.route == expected_route else f"✗ (expected {expected_route})"
    
    print(f"\nTurn {i}: {msg[:70]}..." if len(msg) > 70 else f"\nTurn {i}: {msg}")
    print(f"  Route       : {resp.route}" + (f" / {resp.action}" if resp.action else "") + f"  {match}")
    print(f"  Memory ctx  : {ctx_lines} lines (grows each turn)")
    print(f"  Latency     : {resp.latency_ms}ms")
    print(f"  Answer      : {resp.answer[:300]}{'...' if len(resp.answer) > 300 else ''}")
    print("─" * 72)

print()
logger.success("All 7 turns complete — 3 sub-agents exercised.")

---
## Section 9 — State Inspection

With LangGraph you can inspect exactly what was in `AgentState` at any point in the graph. This is invaluable for debugging — you can see what memory each node received, what the supervisor decided, and what the tool returned.

In [ ]:
# ── Run a single turn and capture the raw state ─────────────────
# We invoke the graph directly (bypassing agent.chat()) to inspect state

from langchain_core.messages import HumanMessage
from agents.state import AgentState

INSPECT_SESSION = "nb02-inspect"
test_message = "What cardiac medications should I know about at Nawaloka Hospital?"

initial_state = {
    "messages":       [HumanMessage(content=test_message)],
    "user_id":        FULL_USER,
    "session_id":     INSPECT_SESSION,
    "memory_context": None,
    "semantic_facts": None,
    "route_decision": None,
    "tool_output":    None,
    "final_answer":   None,
    "should_distill": None,
}

final_state = agent.graph.invoke(initial_state)

print("=" * 72)
print("Final AgentState after full graph traversal:")
print("=" * 72)

print(f"\n[route_decision]")
rd = final_state.get("route_decision", {})
if rd:
    print(f"  route      : {rd.get('route')}")
    print(f"  action     : {rd.get('action')}")
    print(f"  confidence : {rd.get('confidence')}")
    print(f"  reasoning  : {rd.get('reasoning', '')[:120]}")

print(f"\n[memory_context] (first 300 chars):")
mc = final_state.get("memory_context") or "(empty)"
print(f"  {mc[:300]}")

print(f"\n[semantic_facts] ({len(final_state.get('semantic_facts') or [])} facts):")
for f in (final_state.get("semantic_facts") or [])[:3]:
    print(f"  • {f.get('text', '')}  [tags: {f.get('tags', [])}]")

print(f"\n[tool_output] (first 200 chars):")
to = final_state.get("tool_output") or "(none)"
print(f"  {to[:200]}")

print(f"\n[final_answer]:")
fa = final_state.get("final_answer") or "(none)"
print(f"  {fa[:400]}")
print("=" * 72)

---
## Section 10 — Multi-Route: LangGraph Fan-Out

This is the **key new capability**. When a user asks a compound question
containing multiple independent intents, the router returns multiple routes
and LangGraph **fans out to parallel agent nodes**.

**Data flow for a multi-route query:**
1. `recall_node` → loads memory (same as single-route)
2. `supervisor_node` → router detects 2 intents → returns `route_decisions: [{crm}, {rag}]`
3. `supervisor_routing()` returns `["admin", "clinical"]` → **LangGraph fan-out**
4. `admin_agent_node` + `clinical_agent_node` run **in parallel**
5. `merge_responses_node` → combines both outputs via a single synthesiser LLM call
6. `save_memory_node` → stores the merged response

**Latency:** `max(agent1, agent2) + synthesis` — NOT `agent1 + agent2 + synthesis`

For single-route queries, `merge_responses_node` is a **pass-through** (no extra LLM call).


In [ ]:
# ── Multi-Route Demo: Compound question hitting CRM + RAG ────────
resp_multi = agent.chat(
    user_message="Can you check my upcoming appointments and also tell me about the hospital's infection control guidelines?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show(resp_multi, "MULTI-ROUTE — CRM + RAG fan-out")

print(f"\nRoutes taken: {resp_multi.routes}")
print(f"Multi-route: {len(resp_multi.routes) > 1}")
print(f"Primary route: {resp_multi.route}")

In [ ]:
# ── Verify single-route still works identically ─────────────────
resp_single = agent.chat(
    user_message="What are the current visiting hours?",
    user_id=USER_ID, session_id=SESSION_ID,
)
show(resp_single, "SINGLE-ROUTE — no fan-out, merge passes through")

print(f"\nRoutes taken: {resp_single.routes}")
print(f"Multi-route: {len(resp_single.routes) > 1}")
print("No extra LLM call — merge_responses was a pass-through.")

---
## Section 10 — LangFuse: Per-Node Observability

In Week 07, LangFuse produced one trace per `.chat()` call with nested spans. In Week 08, each **LangGraph node** emits its own span, giving you a cleaner waterfall:

```
agent_chat  (trace)
  ├─ recall_node          (span)       ← pgvector query time, facts retrieved
  ├─ supervisor_node      (generation) ← gpt-4o-mini: routing decision, tokens, cost
  ├─ clinical_agent_node  (span)
  │    └─ rag_search      (span)       ← Qdrant KB retrieval, cache HIT/MISS
  │    └─ synthesise      (generation) ← gemini-2.5-flash: answer, tokens, cost
  └─ save_memory_node     (span)
       └─ distill_facts   (generation) ← llama-3.1-8b: fact extraction (if triggered)
```

This makes it easy to identify:
- Which node is the bottleneck (latency)
- Which LLM call costs the most (tokens)
- Whether the CAG cache is working (cache HIT vs MISS)
- What memory context each sub-agent received

In [ ]:
from infrastructure.observability import flush

flush()  # Flush all pending LangFuse events

host = os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com")
print(f"LangFuse dashboard: {host}")
print()
print("What to look for:")
print("  Traces → each .chat() call is one trace")
print("  Spans  → recall_node, supervisor_node, [admin|clinical|direct]_agent_node")
print("  Generations → gpt-4o-mini (routing), gemini (synthesis), llama (distillation)")
print("  Metadata → route_decision, facts_recalled, cache_hit, tokens, cost")

---
## Week 07 vs Week 08 — Side-by-Side

| Aspect | Week 07 (imperative) | Week 08 (LangGraph) |
|---|---|---|
| **Orchestration** | 6 private methods in `AgentOrchestrator` | 6 nodes in `StateGraph` |
| **State** | Implicit — function args | Explicit `AgentState` TypedDict |
| **Routing** | `if/elif` in `_dispatch()` | Compiled conditional edges |
| **Sub-agents** | One synthesiser, role via prompt | 3 persona nodes with guardrails |
| **Memory in context** | Collapsed `str` before any agent | `list[dict]` — each node selects |
| **Visualization** | None | `draw_mermaid()` |
| **Debugging** | loguru logs | State streaming + LangFuse node spans |
| **Adding capabilities** | Refactor `chat()` method | `add_node()` + `add_edge()` |
| **Parallelism** | Not possible | Fan-out edges (future-ready) |

### What's Unchanged

The entire **foundation layer** is identical — memory system, RAG services, CRM tools, infrastructure, and the knowledge base. The LangGraph upgrade is surgical: only the orchestration layer changed.

```
memory/     ← identical (ST, LT, Episodic, Procedural)
services/   ← identical (CAG, CRAG, RAG)
tools/      ← identical (CRM, RAG, Web)
infra/      ← identical (config, db, llm, observability)
data/       ← identical (12 hospital KB documents)
agents/orchestrator.py ← CHANGED: methods → nodes
agents/state.py        ← NEW: AgentState TypedDict
```

---
## Summary

| Component | Role | Key File |
|---|---|---|
| **StateGraph** | Compiled directed graph — nodes + edges | `agents/orchestrator.py` |
| **AgentState** | Shared TypedDict flowing through all nodes | `agents/state.py` |
| **recall_node** | Pulls ST turns + LT facts, populates state | `agents/orchestrator.py` |
| **supervisor_node** | QueryRouter → `route_decision` | `agents/orchestrator.py` |
| **admin_agent_node** | CRM Coordinator persona + `CRMTool` | `agents/orchestrator.py` |
| **clinical_agent_node** | Clinical Knowledge persona + `RAGTool` + selective LT injection | `agents/orchestrator.py` |
| **direct_agent_node** | Concierge persona + `WebSearchTool` fallback | `agents/orchestrator.py` |
| **save_memory_node** | ST write + conditional LT distillation | `agents/orchestrator.py` |
| **CAGCache** | Qdrant semantic cache (KNN-1, threshold=0.90) | `services/chat_service/cag_cache.py` |
| **LangFuse** | Per-node traces, tokens, cost, latency waterfall | `infrastructure/observability.py` |